In [2]:
import pandas as pd

data=pd.read_csv("IMDB Dataset.csv")

print(data.head())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [ ]:
import re
import nltk
from nltk.tokenize import word_tokenize,sent_tokenize

nltk.download('punkt_tab')

def preprocess(text):
    text=text.lower()
    text=re.sub(r"https?://\S+|www\.\S+", "", text)
    text=text.replace("isn't","is not")
    text=text.replace("wasn't","was not")
    text=text.replace("aren't","are not")
    text=text.replace("can't","cannot")
    text=text.replace("couldn't","could not")
    text=text.replace("didn't","did not")
    text=text.replace("doesn't","does not")
    text=text.replace("don't","do not")
    text=text.replace("hadn't","had not")
    text=text.replace("hasn't","has not")
    text=text.replace("haven't","have not")

    text=text.replace(":)","happy")
    text=text.replace(":(","sad")

    tokens=word_tokenize(text)
    tokens=[word for word in tokens if word.isalpha()]
    return tokens

tokens=[]
for review in data["review"]:
    result=preprocess(review)
    tokens.append(result)
data["tokens"]=tokens

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\dipti\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
from nltk.stem import PorterStemmer,SnowballStemmer
stemmer=PorterStemmer()

def stem_tokens(tokens):
    stemmed_tokens=[]
    for word in tokens:
        stemmed=stemmer.stem(word)
        stemmed_tokens.append(stemmed)
    return stemmed_tokens

stemmed_tokens=[]
for tokens in data["tokens"]:
    stemmed=stem_tokens(tokens)
    stemmed_tokens.append(stemmed)
data["stemmed"]=stemmed_tokens
print(data.head())

                                              review sentiment  \
0  One of the other reviewers has mentioned that ...  positive   
1  A wonderful little production. <br /><br />The...  positive   
2  I thought this was a wonderful way to spend ti...  positive   
3  Basically there's a family where a little boy ...  negative   
4  Petter Mattei's "Love in the Time of Money" is...  positive   

                                              tokens  \
0  [one, of, the, other, reviewers, has, mentione...   
1  [a, wonderful, little, production, br, br, the...   
2  [i, thought, this, was, a, wonderful, way, to,...   
3  [basically, there, a, family, where, a, little...   
4  [petter, mattei, love, in, the, time, of, mone...   

                                             stemmed  
0  [one, of, the, other, review, ha, mention, tha...  
1  [a, wonder, littl, product, br, br, the, film,...  
2  [i, thought, thi, wa, a, wonder, way, to, spen...  
3  [basic, there, a, famili, where, a, littl, 

In [ ]:
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
nltk.download("omw-1.4")

lemmatizer=WordNetLemmatizer()

def lemmatize_tokens(tokens):
    lemmatized_tokens=[]
    for word in tokens:
        lemmatized=lemmatizer.lemmatize(word,pos="v")
        lemmatized_tokens.append(lemmatized)
    return lemmatized_tokens

lemmatized_tokens=[]
for tokens in data["tokens"]:
    lemmatized=lemmatize_tokens(tokens)
    lemmatized_tokens.append(lemmatized)
data["lemmatized"]=lemmatized_tokens
print(data.head())

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\dipti\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\dipti\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


                                              review sentiment  \
0  One of the other reviewers has mentioned that ...  positive   
1  A wonderful little production. <br /><br />The...  positive   
2  I thought this was a wonderful way to spend ti...  positive   
3  Basically there's a family where a little boy ...  negative   
4  Petter Mattei's "Love in the Time of Money" is...  positive   

                                              tokens  \
0  [one, of, the, other, reviewers, has, mentione...   
1  [a, wonderful, little, production, br, br, the...   
2  [i, thought, this, was, a, wonderful, way, to,...   
3  [basically, there, a, family, where, a, little...   
4  [petter, mattei, love, in, the, time, of, mone...   

                                             stemmed  \
0  [one, of, the, other, review, ha, mention, tha...   
1  [a, wonder, littl, product, br, br, the, film,...   
2  [i, thought, thi, wa, a, wonder, way, to, spen...   
3  [basic, there, a, famili, where, a, lit

In [ ]:
from collections import Counter

unigrams=Counter()
bigrams=Counter()

for token in data["tokens"]:
    unigrams.update(token)

print(unigrams.most_common(10))

for tokens in data["tokens"]:
    pairs=zip(tokens[:-1],tokens[1:])
    bigrams.update(pairs)

print(bigrams.most_common(10))

[('the', 665362), ('and', 323085), ('a', 321242), ('of', 288531), ('to', 267045), ('is', 216942), ('br', 201931), ('it', 188034), ('in', 185262), ('i', 173680)]
[(('br', 'br'), 101233), (('of', 'the'), 77216), (('in', 'the'), 50367), (('this', 'movie'), 30963), (('and', 'the'), 26845), (('the', 'film'), 26269), (('is', 'a'), 26080), (('to', 'the'), 23772), (('the', 'movie'), 23403), (('to', 'be'), 23270)]
0.030684756215878878


In [ ]:
V=len(unigrams)

def bigram_probability(prev_word,word):
    prob=(bigrams[prev_word,word]+1)/(unigrams[prev_word]+V)
    return prob

print(bigram_probability("the","movie"))
print(bigram_probability("of","the"))

0.030684756215878878
0.2000995094495106


In [ ]:
import math
def sentence_probability(sentence):
    total=0.0
    for i in range(1,len(sentence)):
        p=bigram_probability(sentence[i-1],sentence[i])
        total+=math.log(p)
    return total

print(sentence_probability(["the","movie","was","good"]))

-12.838387293182214


In [ ]:
def perplexity(sentence):
    logp=sentence_probability(sentence)
    return math.exp(-logp/len(sentence)-1)

print(perplexity(["the","movie","was","good"]))

9.112041889333213


In [ ]:
#NOT DONE YET


import random

def generate(start, length=10, seed=42):
    random.seed(seed)
    words = [start]

    for _ in range(length - 1):
        previous = words[-1]

        candidates = [
            word for (prev, word) in bigrams
            if prev == previous
        ]

        if not candidates:
            break

        weights = [
            bigram_probability(previous, word)
            for word in candidates
        ]

        next_word = random.choices(
            candidates, weights=weights
        )[0]

        words.append(next_word)

    return " ".join(words)

print(generate("the"))

the fifties the romantic br when one mademouiselle choule imagining


In [ ]:
from nltk import pos_tag

nltk.download("averaged_perceptron_tagger_eng")

def pos_tagging(tokens):
    tags=pos_tag(tokens)
    return tags

pos_tags=[]
for tokens in data["tokens"]:
    tags=pos_tagging(tokens)
    pos_tags.append(tags)

data["pos"]=pos_tags
print(data.head())

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\dipti\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


                                              review sentiment  \
0  One of the other reviewers has mentioned that ...  positive   
1  A wonderful little production. <br /><br />The...  positive   
2  I thought this was a wonderful way to spend ti...  positive   
3  Basically there's a family where a little boy ...  negative   
4  Petter Mattei's "Love in the Time of Money" is...  positive   

                                              tokens  \
0  [one, of, the, other, reviewers, has, mentione...   
1  [a, wonderful, little, production, br, br, the...   
2  [i, thought, this, was, a, wonderful, way, to,...   
3  [basically, there, a, family, where, a, little...   
4  [petter, mattei, love, in, the, time, of, mone...   

                                             stemmed  \
0  [one, of, the, other, review, ha, mention, tha...   
1  [a, wonder, littl, product, br, br, the, film,...   
2  [i, thought, thi, wa, a, wonder, way, to, spen...   
3  [basic, there, a, famili, where, a, lit

In [ ]:
import nltk

grammar = nltk.CFG.fromstring("""
S -> NP VP
NP -> Det N
VP -> V NP
Det -> 'the' | 'a'
N -> 'Dipti' | 'movie'
V -> 'likes' | 'sees'
""")

parser=nltk.ChartParser(grammar)

sentence="the Dipti likes a movie".split()

for tree in parser.parse(sentence):
    print(tree)
    tree.pretty_print()

(S (NP (Det the) (N Dipti)) (VP (V likes) (NP (Det a) (N movie))))
                S                
      __________|____             
     |               VP          
     |           ____|___         
     NP         |        NP      
  ___|____      |     ___|____    
Det       N     V   Det       N  
 |        |     |    |        |   
the     Dipti likes  a      movie



In [ ]:
import spacy

nlp=spacy.load("en_core_web_sm")

text="the movie was good but the acting was bad"

doc=nlp(text)

for word in doc:
    print(word.text,word.pos_,word.dep_,"head: ",word.head.text)

the DET det head:  movie
movie NOUN nsubj head:  was
was AUX ROOT head:  was
good ADJ acomp head:  was
but CCONJ cc head:  was
the DET det head:  acting
acting NOUN nsubj head:  was
was AUX conj head:  was
bad ADJ acomp head:  was


In [ ]:
doc=nlp("the movie was good but the acting was bad")

for word in doc:
    if word.dep_=="acomp":
        for child in word.head.children:
            if child.dep_=="nsubj":
                print("aspect:",child.text,"opinion:",word.text)

aspect: movie opinion: good
aspect: acting opinion: bad


In [ ]:
from nltk.corpus import wordnet as wn 

nltk.download("wordnet")
nltk.download("omw-1.4")

synsets=wn.synsets("duck")

for syn in synsets[:5]:
    print(syn.name())
    print(syn.definition())

duck.n.01
small wild or domesticated web-footed broad-billed swimming bird usually having a depressed body and short legs
duck.n.02
(cricket) a score of nothing by a batsman
duck.n.03
flesh of a duck (domestic or wild)
duck.n.04
a heavy cotton fabric of plain weave; used for clothing and tents
duck.v.01
to move (the head or body) quickly downwards or away


[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\dipti\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\dipti\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
word1=wn.synsets("book")[0]
word2=wn.synsets("pen")[0]

similarity=word1.wup_similarity(word2)

print("similarity: ",similarity)

similarity:  0.5263157894736842


In [ ]:
from nltk.wsd import lesk

sentence="i went to the bank to deposit money" 

words=word_tokenize(sentence)

meaning=lesk(words,"bank")

print("selected meaning: ",meaning)
print("definition: ",meaning.definition())

selected meaning:  Synset('depository_financial_institution.n.01')
definition:  a financial institution that accepts deposits and channels the money into lending activities


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

texts=data["review"].tolist()

vectorizer=TfidfVectorizer(max_features=5000)

X=vectorizer.fit_transform(texts)

print("tf idf matrix shape: ",X.shape)

tf idf matrix shape:  (50000, 5000)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

text="it was a good movie"

words=word_tokenize(text)

vectorizer=TfidfVectorizer()

X=vectorizer.fit_transform(words)

print(X)
print(X.shape)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4 stored elements and shape (5, 4)>
  Coords	Values
  (0, 1)	1.0
  (1, 3)	1.0
  (3, 0)	1.0
  (4, 2)	1.0
(5, 4)


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

query="good movie"

query_vector=vectorizer.transform([query])

scores=cosine_similarity(query_vector,X)[0]

print(scores[:10])


[0.         0.         0.         0.05285947 0.06711743 0.05140824
 0.         0.         0.         0.10323399]


In [ ]:
top_k = 3

indices = scores.argsort()[::-1][:top_k]

for i in indices:
    print("Score:", round(scores[i], 3))
    print("Review:", data.iloc[i]["review"][:300])
    print()

Score: 0.547
Review: Englar Alheimsins are very good movie. She happen on a mental home in Iceland. Ingvar E. Sigurdsson is in a leading role and is good. Other good actors in this movie are Baltasar Kormákur and Bjorn Jorundur. I like this movie she is very good. I voice with this movie.

Score: 0.443
Review: Good story. Good script. Good casting. Good acting. Good directing. Good art direction. Good photography. Good sound. Good editing. Good everything. Put it all together and you end up with good entertainment.<br /><br />The shame of it is that there aren't nearly enough films of this caliber being m

Score: 0.441
Review: I don't get this. The movie obviously has a pretty good budget. It has very good cinematography. It has nice pacing, good editing and pretty good directing too. Then WHY OH WHY didn't they hire someone to do a final rewrite of the script so it would not be so damn cheesy and WHY OH WHY did they hire



In [ ]:
import numpy as np
from collections import Counter

window = 2

pair_counts = Counter()
word_counts = Counter()

for tokens in data["tokens"]:
    for i, word in enumerate(tokens):
        word_counts[word] += 1

        start = max(0, i - window)
        end = min(len(tokens), i + window + 1)

        for j in range(start, end):
            if i != j:
                context_word = tokens[j]
                pair_counts[(word, context_word)] += 1

print(pair_counts.most_common(10))

[(('br', 'br'), 204306), (('of', 'the'), 151472), (('the', 'of'), 151472), (('and', 'the'), 71506), (('the', 'and'), 71506), (('in', 'the'), 67197), (('the', 'in'), 67197), (('is', 'the'), 55721), (('the', 'is'), 55721), (('to', 'the'), 55521)]


In [ ]:
total_pairs = sum(pair_counts.values())

ppmi = {}

for (word, context), count in pair_counts.items():
    pmi = np.log(
        (count * total_pairs) /
        (word_counts[word] * word_counts[context])
    )

    ppmi[(word, context)] = max(0, pmi)

print(list(ppmi.items())[:10])

[(('one', 'of'), np.float64(4.0287100144065935)), (('one', 'the'), np.float64(3.095406891022268)), (('of', 'one'), np.float64(4.0287100144065935)), (('of', 'the'), np.float64(3.5802209819739144)), (('of', 'other'), np.float64(3.000777833189595)), (('the', 'one'), np.float64(3.095406891022268)), (('the', 'of'), np.float64(3.5802209819739144)), (('the', 'other'), np.float64(3.1398332663654314)), (('the', 'reviewers'), np.float64(3.2707492820794344)), (('other', 'of'), np.float64(3.000777833189595))]


In [ ]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=data["tokens"].tolist(),
    vector_size=50,
    window=2,
    min_count=2,
    sg=1,
    seed=42,
    workers=1,
    epochs=20
)

print(model.wv["movie"][:10])

NameError: name 'data' is not defined

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

text = "Priya watched a movie. She loved it."

doc = nlp(text)

# Named entities
print("Named entities:")
for ent in doc.ents:
    print(ent.text, ent.label_)

# Noun phrases
print("\nNoun chunks:")
for chunk in doc.noun_chunks:
    print(chunk.text)

# Pronouns
print("\nPronouns:")
for token in doc:
    if token.pos_ == "PRON":
        print(token.text)

Named entities:

Noun chunks:
Priya
a movie
She
it

Pronouns:
She
it


In [ ]:
gender = {
    "priya": "female",
    "she": "female",
    "he": "male"
}

number = {
    "priya": "singular",
    "she": "singular",
    "he": "singular"
}

def score_pair(name, pronoun, distance):
    score = 0

    # Gender agreement
    if gender.get(name) == gender.get(pronoun):
        score += 2

    # Number agreement
    if number.get(name) == number.get(pronoun):
        score += 2

    # Prefer closer mentions
    score += max(0, 3 - distance)

    return score

print(score_pair("priya", "she", 1))
print(score_pair("priya", "he", 1))

6
4


In [ ]:
chains = [
    ["Priya", "She"],
    ["a movie", "it"]
]

for chain in chains:
    print(" -> ".join(chain))

Priya -> She
a movie -> it


In [ ]:
text = data["review"].iloc[0][:500]

doc = nlp(text)

print("Review:", text)

print("\nNoun chunks:")
for chunk in doc.noun_chunks:
    print(chunk.text)

print("\nPronouns:")
for token in doc:
    if token.pos_ == "PRON":
        print(token.text)

Review: One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ

Noun chunks:
the other reviewers
just 1 Oz episode
you
They
this
exactly what
me.<br
first thing
that
me
Oz
its brutality
scenes
violence
which
right
the word
GO
me
this
a show
This show
no punches
regards
drugs
sex
violence
Its
hardcore
the classic use
the word.<br
/>It
OZ

Pronouns:
you
They
this
what
/>The
that
me
its
which
me
this
Its


In [ ]:
from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)

print(sentiment_model("The movie was excellent!"))
print(sentiment_model("The plot was boring and terrible."))

In [ ]:
for review in data["review"].head(5):
    result = sentiment_model(review[:512])[0]

    print("Review:", review[:100])
    print("Prediction:", result["label"])
    print("Score:", round(result["score"], 3))
    print()

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    data[["review", "sentiment"]],
    test_size=0.2,
    random_state=42,
    stratify=data["sentiment"]
)

print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

In [ ]:
label_map = {
    "negative": 0,
    "positive": 1
}

train_df = train_df.copy()
test_df = test_df.copy()

train_df["label"] = train_df["sentiment"].map(label_map)
test_df["label"] = test_df["sentiment"].map(label_map)

print(train_df[["review", "label"]].head())

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

train_data = Dataset.from_pandas(
    train_df[["review", "label"]].rename(
        columns={"review": "text"}
    ),
    preserve_index=False
)

test_data = Dataset.from_pandas(
    test_df[["review", "label"]].rename(
        columns={"review": "text"}
    ),
    preserve_index=False
)

tokenizer = AutoTokenizer.from_pretrained(
    "google-bert/bert-base-uncased"
)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_data = train_data.map(tokenize, batched=True)
test_data = test_data.map(tokenize, batched=True)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "google-bert/bert-base-uncased",
    num_labels=2
)

In [ ]:
import numpy as np
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(pred):
    logits, labels = pred
    predictions = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(
            labels, predictions,
            zero_division=0
        )
    }

args = TrainingArguments(
    output_dir="./bert_results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    report_to="none",
    save_strategy="no"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_data,
    eval_dataset=test_data,
    compute_metrics=compute_metrics
)

trainer.train()

print(trainer.evaluate())

In [ ]:
import torch

def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    )

    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)

    predicted_class = outputs.logits.argmax(dim=1).item()

    if predicted_class == 1:
        return "positive"
    else:
        return "negative"

print(predict_sentiment("The acting was wonderful"))
print(predict_sentiment("The story was painfully boring"))